# 03 — Deep Learning Fundamentals — Lab

This lab gives you a direct, hands-on implementation of the core ideas from the lecture: fully connected layers, forward propagation, cross-entropy loss, backpropagation, and Adam updates. You will build a small MLP from raw PyTorch tensor operations, train it on MNIST, and verify that it reaches the target performance threshold. The exercise stays within the lesson scope and does not use convolutional or attention-based layers.

## Objectives

- Trace forward and backward passes through a two-hidden-layer network by hand, writing out the matrix multiplications and chain-rule gradient expressions at each layer.
- Implement a multi-layer perceptron from scratch in PyTorch using only tensor operations and state why each activation function and initializer was chosen.
- Select an appropriate loss function for a given task and derive its gradient with respect to the output logits.
- Train the from-scratch MLP to >97% test accuracy on MNIST and explain how each optimizer hyperparameter affects convergence.

## Prerequisites

You should already be comfortable with the lecture material from this lesson, including fully connected layers, activation functions (ReLU, sigmoid, tanh), forward propagation, cross-entropy loss, backpropagation, and Adam updates. You should also know the core ideas from the previous module: what a shallow MLP is, why depth matters, and how a model learns by adjusting weights through optimization.

## Required Software and Packages

- Python 3.10+
- PyTorch
- torchvision

No additional packages are required beyond the standard course environment. If PyTorch and torchvision are already installed in your course environment, no extra installation step is needed.

## Environment Setup

Run the next cell to import the libraries needed for the lab and prepare the MNIST dataset.

In [ ]:
import torch
import torch.nn as nn
from torchvision import datasets, transforms

print(f"PyTorch version: {torch.__version__}")
print("Setup cell executed successfully.")

## Background

A multi-layer perceptron is a stack of affine transforms and nonlinear activations. In a two-layer hidden network, the input is projected into a hidden space, transformed with a nonlinearity, then projected again to class logits. The network must learn weights that minimize the classification loss, and backpropagation provides the exact gradient signal needed to update those weights.

This lab uses the MNIST dataset, whose inputs are 28x28 grayscale images flattened to 784 features. Each example is labeled as one of 10 digits. The target is a class index, the model outputs 10 logits, and the loss is cross-entropy because the task is multiclass classification.

You will implement the MLP using raw tensor operations and `torch.autograd`, rather than using high-level layer wrappers like `nn.Linear` or `nn.Sequential`. This keeps the exercise aligned with the lecture and emphasizes the actual matrix multiplications and gradient flow that power a neural network.

## Exercise Instructions

1. Implement a minimal `MLP` class with `nn.Parameter` weights and explicit matrix multiplies plus ReLU. Confirm the output shape on a single synthetic batch. (3 min)
2. Load MNIST, train the model for 10 epochs using cross-entropy loss and Adam with `lr=1e-3`, and print the training loss each epoch. (5 min)
3. Evaluate the trained model on the MNIST test set and confirm that final test accuracy is at least 97%. (12 min)

In [ ]:
# Starter code: complete the TODOs in the MLP below.

class MLP(nn.Module):
    def __init__(self, input_dim=784, hidden_dim=128, output_dim=10):
        super().__init__()
        # TODO: create nn.Parameter weights for the hidden layer and output layer.
        # Use Xavier or Kaiming-style initialization and shape the tensors correctly.
        self.W1 = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.01)
        self.b1 = nn.Parameter(torch.zeros(hidden_dim))
        self.W2 = nn.Parameter(torch.randn(hidden_dim, output_dim) * 0.01)
        self.b2 = nn.Parameter(torch.zeros(output_dim))

    def forward(self, x):
        # x shape: (batch_size, 784)
        # TODO: compute z1 = x @ W1 + b1, then h1 = ReLU(z1), then logits = h1 @ W2 + b2
        z1 = x @ self.W1 + self.b1
        h1 = torch.relu(z1)
        logits = h1 @ self.W2 + self.b2
        return logits

# Quick shape check: this should produce logits of shape (batch_size, 10)
example_x = torch.randn(8, 784)
model = MLP()
with torch.no_grad():
    logits = model(example_x)
print("Example logits shape:", tuple(logits.shape))

### Step 1: Build the MLP

In [ ]:
# Step 1: implement the class and verify a forward pass.
# Replace the default random initialization with a deliberate initialization strategy.
# Keep the architecture simple: one hidden layer, ReLU activation, and a final linear output.

# TODO: create a model instance, run one synthetic batch through it, and print shape.
# Example:
# model = MLP(input_dim=784, hidden_dim=128, output_dim=10)
# x = torch.randn(16, 784)
# logits = model(x)
# print(logits.shape)

model = MLP(input_dim=784, hidden_dim=128, output_dim=10)
example_x = torch.randn(16, 784)
logits = model(example_x)
print(f"Batch size: {example_x.shape[0]}, logits shape: {tuple(logits.shape)}")

**Expected output:** A tensor whose shape is `(batch_size, 10)`, for example `(16, 10)`. The model should accept a 784-dimensional input and return one logit per class.

### Step 2: Train on MNIST with Cross-Entropy and Adam

In [ ]:
# Step 2: train on MNIST with cross-entropy loss and Adam.
# Complete the dataset setup, dataloader creation, and training loop.

# TODO: define the data transforms, create train_loader, and initialize the model and optimizer.
# Use Adam with lr=1e-3 and cross_entropy loss.

train_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root='.', train=True, download=True, transform=train_transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True)

model = MLP(input_dim=784, hidden_dim=128, output_dim=10)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

for epoch in range(10):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images = images.view(-1, 784)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * images.size(0)

    avg_loss = running_loss / len(train_dataset)
    print(f"Epoch {epoch + 1:02d} | train loss: {avg_loss:.4f}")

**Expected output:** Ten epoch-by-epoch training losses printed in the terminal or notebook output. Early epochs should be relatively high, and later epochs should trend downward as the network learns.

### Step 3: Evaluate on the Test Set and Check the Accuracy Goal

In [ ]:
# Step 3: evaluate on the MNIST test set and compute accuracy.
# Make sure you compute accuracy using the model's logits and the correct labels.

# TODO: define the test dataset and dataloader, run the model in eval mode, and print the final accuracy.

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

test_dataset = datasets.MNIST(root='.', train=False, download=True, transform=test_transform)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=256, shuffle=False)

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images = images.view(-1, 784)
        logits = model(images)
        predictions = logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"Test accuracy: {test_accuracy:.4f}")

**Expected output:** A final test accuracy near or above `0.97` for a correctly trained model. If your code is correct and training is stable, your value should be above the required threshold.

## Comprehension Questions

1. Why does ReLU help gradients flow more cleanly in a deep MLP than sigmoid or tanh, especially when the network is only a few layers deep but still training from scratch?
2. In your own words, how does Adam differ from vanilla SGD, and why do the first- and second-moment estimates matter for convergence on MNIST?

**Write your answers in a markdown cell or in a short note. They are not required for grading, but they help connect what you have implemented to the lecture’s discussion of gradient flow and optimizer behavior.**

In [ ]:
# Verification: check that the model and training loop meet the lesson requirements.
# These assertions are intentionally strict: if they fail, review the earlier steps.

assert isinstance(model, nn.Module), "The model should be a torch.nn.Module instance."
assert hasattr(model, 'W1') and hasattr(model, 'W2'), "The model must define trainable weight parameters."
assert model.W1.shape == (784, 128), f"Hidden weight shape mismatch: {tuple(model.W1.shape)}"
assert model.W2.shape == (128, 10), f"Output weight shape mismatch: {tuple(model.W2.shape)}"
assert test_accuracy >= 0.97, f"Expected test accuracy >= 0.97, got {test_accuracy:.4f}."
print("✓ PASS: The MLP was implemented, trained, and evaluated successfully.")
print(f"  - Final MNIST test accuracy: {test_accuracy:.4f}")
print("  - The model uses explicit tensor operations and ReLU activation in the hidden layer.")
print("  - Cross-entropy loss and Adam were used for training.")

## Optional Challenges

These challenges are clearly optional and are not required to complete the lab.

1. Replace the hidden layer size with 64 or 256 units and compare the training speed and final accuracy. How does capacity change the convergence behavior?
2. Try a different optimizer, such as SGD with momentum, and compare the curve of training loss to Adam. Which hyperparameters seem most important for stable learning?

## Completion Criteria

You have successfully completed this lab when:

1. The MLP class is implemented using `nn.Parameter` weights and explicit tensor operations, with ReLU in the hidden layer and a 10-class output.
2. The training loop runs for 10 epochs using cross-entropy loss and Adam with `lr=1e-3` and prints a loss for each epoch.
3. The test-set evaluation cell reports a valid final accuracy on MNIST.
4. The verification cell prints a clear `PASS` result and confirms that the final accuracy is at least 97%.
5. You can explain why the hidden-layer activation and optimizer choice matter for gradient flow and convergence.